# Video Scraping Pipeline

An asynchronous, two-stage video scraping and downloading pipeline:
- **Stage 1 (Scout):** `Crawl4AI` + local **Gemma-4 E4B** (via Ollama) automatically searches the web and extracts video URLs.
- **Stage 2 (Download):** `yt-dlp` downloads the discovered videos.
- **Stage 3 (Manager):** An `asyncio` orchestrator with robust error handling.

**Just provide a search prompt** (e.g. *"helmet violations Vietnam dashcam"*) and the agent
will automatically discover and download matching videos.

## 1. Environment Setup

In [1]:
import subprocess, sys, os

# System libraries required by Playwright / Chromium on Kaggle
subprocess.run(
    ["apt-get", "update", "-qq"],
    capture_output=True,
)
subprocess.run(
    ["apt-get", "install", "-y", "-qq",
     "libnss3", "libnspr4", "libatk1.0-0", "libatk-bridge2.0-0",
     "libcups2", "libdrm2", "libxkbcommon0", "libxcomposite1",
     "libxdamage1", "libxfixes3", "libxrandr2", "libgbm1",
     "libasound2", "libpango-1.0-0", "libcairo2"],
    capture_output=True,
)
print("System dependencies installed.")

# Python packages
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "crawl4ai", "yt-dlp", "pydantic", "nest_asyncio",
])
print("Python packages installed.")

# Playwright browsers (Chromium) for crawl4ai
subprocess.run(
    [sys.executable, "-m", "playwright", "install", "chromium"],
    capture_output=True,
)
subprocess.run(
    [sys.executable, "-m", "playwright", "install-deps", "chromium"],
    capture_output=True,
)
print("Playwright Chromium installed.")

System dependencies installed.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.9/501.9 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ypy-websocket 0.8.4 requires aiofiles<23,>=22.1.0, but you have aiofiles 25.1.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.7 which is incompatible.
pydrive2 1.21.3 requires pyOpenSSL<=24.2.1,>=19.1.0, but you have pyopenssl 26.0.0 which is incompatible.
gradio 5.50.0 requires aiofiles<25.0,>=22.0, but you have aiofiles 25.1.0 which is incompatible.


Python packages installed.
Playwright Chromium installed.


In [2]:
# Install Ollama
!apt-get update && apt-get install -y zstd && apt-get install -y nodejs
!curl -fsSL https://ollama.com/install.sh | sh
print("Ollama installed.")

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



The following NEW packages will be insta

In [3]:
import time, subprocess

# Ollama listens on http://localhost:11434 and auto-detects the T4 GPU.
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)
print(f"Ollama server started (pid={ollama_proc.pid})")

# gemma4:e4b — effective 4B params, ~9.6 GB download, ~6 GB VRAM on T4
!ollama pull gemma4:e4b
!nvidia-smi

Ollama server started (pid=2881)
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 4c27e0f5b5ad:   0% ▕                  ▏  21 MB/9.6 GB                  pulling manifest 
pulling 4c27e0f5b5ad:   1% ▕                  ▏ 106 MB/9.6 GB                  pulling manifest 
pulling 4c27e0f5b5ad:   2% ▕                  ▏ 196 MB/9.6 GB                  pulling manifest 
pulling 4c27e0f5b5ad:   2% ▕                  ▏ 233 MB/9.6 GB                  pulling manifest 
pulling 4c27e0f5b5ad:   3% ▕                  ▏ 328 MB/9.6 GB                  pulling manifest 
pulling 4c27e0f5b5ad:   4% ▕                  ▏ 421 MB/9.

## 2. Imports & Configuration

In [4]:
from __future__ import annotations

import asyncio
import json
import logging
import os
import re
import subprocess
import time
from pathlib import Path
from urllib.parse import quote_plus

import nest_asyncio
from pydantic import BaseModel, Field

nest_asyncio.apply()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("pipeline")

# ---------------------------------------------------------------------------
# Pydantic schema — the JSON shape Gemma-4 must return per video.
# ---------------------------------------------------------------------------
class VideoExtractionSchema(BaseModel):
    video_title: str = Field(description="Title or short description of the video")
    raw_video_url: str = Field(description="Direct or watchable URL of the video")
    description: str = Field(description="Brief description of the video content")

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
OLLAMA_MODEL = "gemma4:e4b"
OLLAMA_HOST  = "http://localhost:11434"
DOWNLOAD_DIR = "./downloads"

print(f"Model  : {OLLAMA_MODEL}")
print(f"Host   : {OLLAMA_HOST}")
print(f"Output : {DOWNLOAD_DIR}")

Model  : gemma4:e4b
Host   : http://localhost:11434
Output : ./downloads


## 3. Stage 1 — The Scout (Crawl4AI + Gemma-4)

Given a **search prompt** (e.g. *"helmet violations Vietnam"*), the scout:
1. Builds Google / YouTube search URLs automatically.
2. Crawls each search-result page with `AsyncWebCrawler`.
3. Asks Gemma-4 to extract video metadata as strict JSON.

In [5]:
from crawl4ai import (
    AsyncWebCrawler,
    BrowserConfig,
    CrawlerRunConfig,
    CacheMode,
    LLMConfig,
    LLMExtractionStrategy,
)


SCOUT_INSTRUCTION = (
    "You are analysing a web page to find videos related to helmet violations "
    "on Vietnamese roads — motorcycle riders or passengers NOT wearing helmets. "
    "Extract EVERY video you can find on this page. For each video return: "
    "video_title, raw_video_url (a direct or watchable link such as a YouTube URL), "
    "and a short description. "
    "Output ONLY strict JSON — an array of objects matching the schema. "
    "If no relevant videos exist, return an empty array []."
)


def build_search_urls(prompt: str) -> list[str]:
    """Turn a free-text prompt into search-engine URLs to crawl."""
    q = quote_plus(prompt)
    # Vietnamese-language variant
    vn_keywords = [
        "vi pham khong doi mu bao hiem",
        "camera giao thong khong mu bao hiem",
        "khong doi mu bao hiem xe may",
    ]
    urls = [
        # YouTube search pages
        f"https://www.youtube.com/results?search_query={q}",
    ]
    for kw in vn_keywords:
        urls.append(
            f"https://www.youtube.com/results?search_query={quote_plus(kw)}"
        )
    # Google search (video tab)
    urls.append(
        f"https://www.google.com/search?q={q}&tbm=vid"
    )
    return urls


async def scout_videos(url: str) -> list[dict]:
    """Crawl *url* and use Gemma-4 to extract video metadata as JSON."""

    llm_cfg = LLMConfig(
        provider=f"ollama/{OLLAMA_MODEL}",
        base_url=OLLAMA_HOST,
    )

    llm_strategy = LLMExtractionStrategy(
        llm_config=llm_cfg,
        schema=VideoExtractionSchema.model_json_schema(),
        extraction_type="schema",
        instruction=SCOUT_INSTRUCTION,
        chunk_token_threshold=4000,
        overlap_rate=0.1,
        apply_chunking=True,
        extra_args={"temperature": 0.0, "max_tokens": 4000},
    )

    crawl_cfg = CrawlerRunConfig(
        extraction_strategy=llm_strategy,
        cache_mode=CacheMode.BYPASS,
        word_count_threshold=10,
        page_timeout=80000,
    )

    browser_cfg = BrowserConfig(headless=True)

    log.info("Scouting: %s", url)

    async with AsyncWebCrawler(config=browser_cfg) as crawler:
        result = await crawler.arun(url=url, config=crawl_cfg)

    if not result.extracted_content:
        log.warning("No extracted content from %s", url)
        return []

    try:
        data = json.loads(result.extracted_content)
    except json.JSONDecodeError as exc:
        log.warning("Gemma-4 returned invalid JSON for %s: %s", url, exc)
        return []

    if isinstance(data, dict):
        data = [data]
    if not isinstance(data, list):
        log.warning("Unexpected extraction shape from %s: %s", url, type(data))
        return []

    valid = []
    for item in data:
        if isinstance(item, dict) and item.get("raw_video_url"):
            valid.append(item)
    log.info("Found %d video(s) from %s", len(valid), url)
    return valid


print("build_search_urls() and scout_videos() defined.")

build_search_urls() and scout_videos() defined.


## 4. Stage 2 — The Downloader (yt-dlp)

In [6]:
import yt_dlp


def _sanitize_filename(name: str) -> str:
    """Remove characters unsafe for filenames."""
    return re.sub(r'[\\/*?:"<>|]', "", name).strip()[:120] or "video"


def download_video(
    video_title: str, video_url: str, output_dir: str = DOWNLOAD_DIR
) -> str | None:
    """Download a single video with yt-dlp.

    Returns the local file path on success, or None on failure.
    """
    os.makedirs(output_dir, exist_ok=True)
    safe_title = _sanitize_filename(video_title)
    out_template = os.path.join(output_dir, f"{safe_title}.%(ext)s")

    ydl_opts = {
        "outtmpl": out_template,
        "format": "bestvideo+bestaudio/best",
        "merge_output_format": "mp4",
        "noplaylist": True,
        "socket_timeout": 30,
        "quiet": False,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(video_url, download=True)
            filename = ydl.prepare_filename(info)
            final = Path(filename).with_suffix(".mp4")
            if not final.exists():
                candidates = list(Path(output_dir).glob(f"{safe_title}.*"))
                final = candidates[0] if candidates else final
            log.info("Downloaded -> %s", final)
            return str(final)

    except yt_dlp.utils.DownloadError as exc:
        log.warning("yt-dlp DownloadError for '%s': %s", video_url, exc)
        return None
    except Exception as exc:
        log.warning("Unexpected download error for '%s': %s", video_url, exc)
        return None


print("download_video() defined.")

download_video() defined.


## 5. Stage 3 — The Pipeline Manager

Coordinates the full workflow:

1. Take a **search prompt** from the user.
2. Automatically build search URLs (YouTube + Google Videos).
3. Crawl each page, ask Gemma-4 to extract video links.
4. Download every discovered video with yt-dlp.

Error handling:
- If Gemma-4 returns invalid JSON → skip that page.
- If yt-dlp encounters a `DownloadError` → skip that video, continue.

In [7]:
async def run_pipeline(prompt: str) -> list[dict]:
    """End-to-end: build search URLs from *prompt*, scout them, download videos."""

    search_urls = build_search_urls(prompt)
    log.info("Search prompt: %s", prompt)
    log.info("Generated %d search URLs to crawl.", len(search_urls))
    for u in search_urls:
        log.info("  %s", u)

    all_videos: list[dict] = []
    results: list[dict] = []

    # --- Stage 1: Scout ---
    for url in search_urls:
        try:
            videos = await scout_videos(url)
            all_videos.extend(videos)
        except Exception as exc:
            log.error("Scout failed for %s: %s", url, exc)

    # Deduplicate by raw_video_url
    seen_urls: set[str] = set()
    unique_videos: list[dict] = []
    for v in all_videos:
        vurl = v.get("raw_video_url", "")
        if vurl and vurl not in seen_urls:
            seen_urls.add(vurl)
            unique_videos.append(v)

    log.info("=" * 50)
    log.info("Scouting complete. %d unique video(s) found.", len(unique_videos))
    log.info("=" * 50)

    if not unique_videos:
        log.warning("Nothing to download.")
        return results

    # --- Stage 2: Download ---
    for v in unique_videos:
        title = v.get("video_title", "untitled")
        vurl  = v.get("raw_video_url", "")
        desc  = v.get("description", "")
        log.info("Downloading: %s  (%s)", title, vurl)

        local_path = download_video(title, vurl)
        results.append({
            "video_title": title,
            "raw_video_url": vurl,
            "description": desc,
            "local_path": local_path,
            "status": "ok" if local_path else "failed",
        })

    # --- Summary ---
    ok = sum(1 for r in results if r["status"] == "ok")
    log.info("=" * 50)
    log.info("PIPELINE COMPLETE")
    log.info("  Search prompt    : %s", prompt)
    log.info("  Pages crawled    : %d", len(search_urls))
    log.info("  Videos discovered: %d", len(unique_videos))
    log.info("  Downloads OK     : %d", ok)
    log.info("  Downloads failed : %d", len(results) - ok)
    log.info("  Output directory : %s", DOWNLOAD_DIR)
    log.info("=" * 50)

    return results


print("run_pipeline() defined.")

run_pipeline() defined.


## 6. Run the Pipeline

Change the `SEARCH_PROMPT` below to tell the agent what videos to find.
It will automatically search YouTube and Google, extract video URLs via
Gemma-4, and download them.

In [8]:
# ============================================================
# CHANGE THIS PROMPT to search for different videos.
# The agent builds search URLs automatically from this text.
# ============================================================
SEARCH_PROMPT = "helmet violation Vietnam motorcycle dashcam no helmet"

results = asyncio.run(run_pipeline(SEARCH_PROMPT))

17:15:46  INFO      Search prompt: helmet violation Vietnam motorcycle dashcam no helmet
17:15:46  INFO      Generated 5 search URLs to crawl.
17:15:46  INFO        https://www.youtube.com/results?search_query=helmet+violation+Vietnam+motorcycle+dashcam+no+helmet
17:15:46  INFO        https://www.youtube.com/results?search_query=vi+pham+khong+doi+mu+bao+hiem
17:15:46  INFO        https://www.youtube.com/results?search_query=camera+giao+thong+khong+mu+bao+hiem
17:15:46  INFO        https://www.youtube.com/results?search_query=khong+doi+mu+bao+hiem+xe+may
17:15:46  INFO        https://www.google.com/search?q=helmet+violation+Vietnam+motorcycle+dashcam+no+helmet&tbm=vid
17:15:46  INFO      Scouting: https://www.youtube.com/results?search_query=helmet+violation+Vietnam+motorcycle+dashcam+no+helmet


[INIT].... → Crawl4AI 0.8.6 

[FETCH]... ↓ https://www.youtube.com/results?search_query=helmet+violation+Vietnam+motorcycle+dashcam+no+helmet   |
✓ | ⏱: 3.35s 

[SCRAPE].. ◆ https://www.youtube.com/results?search_query=helmet+violation+Vietnam+motorcycle+dashcam+no+helmet   |
✓ | ⏱: 0.15s 

17:15:56 - LiteLLM:INFO: utils.py:3889 - 
LiteLLM completion() model= gemma4:e4b; provider = ollama
17:15:56  INFO      
LiteLLM completion() model= gemma4:e4b; provider = ollama


...

ERROR: [Instagram] DWzIEk2jRqI: Requested content is not available, rate-limit reached or login required. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies
17:28:01  WARNING   yt-dlp DownloadError for 'https://www.instagram.com/reel/DWzIEk2jRqI/': ERROR: [Instagram] DWzIEk2jRqI: Requested content is not available, rate-limit reached or login required. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies
17:28:01  INFO      Downloading: Two-wheeler or Tempo? 6 People on ONE Moped – Shocking...  (https://www.facebook.com/ind2day/posts/-two-wheeler-or-tempo-6-people-on-one-moped-shocking-roadsafety-nightmarefamily-/1616336553835141/)


[generic] Extracting URL: https://www.facebook.com/ind2day/posts/-two-wheeler-or-tempo-6-people-on-one-moped-shocking-roads...y-/1616336553835141/
[generic] 1616336553835141: Downloading webpage
[redirect] Following redirect to https://www.facebook.com/ind2day/videos/-two-wheeler-or-tempo-6-people-on-one-moped-shocking-roadsafety-nightmarefamily-/1561250638305790/
[facebook] Extracting URL: https://www.facebook.com/ind2day/videos/-two-wheeler-or-tempo-6-people-on-one-moped-shocking-road...y-/1561250638305790/
[facebook] 1561250638305790: Downloading webpage
[info] 1561250638305790: Downloading 1 format(s): 1308621781214936v+1904223323573432a
[download] Destination: ./downloads/Two-wheeler or Tempo 6 People on ONE Moped – Shocking....f1308621781214936v.mp4
[download] 100% of   21.77MiB in 00:00:00 at 38.95MiB/s  
[download] Destination: ./downloads/Two-wheeler or Tempo 6 People on ONE Moped – Shocking....f1904223323573432a.m4a
[download] 100% of  747.54KiB in 00:00:00 at 5.13MiB/s   
[M

17:28:04  INFO      Downloaded -> downloads/Two-wheeler or Tempo 6 People on ONE Moped – Shocking....mp4
17:28:04  INFO      Downloading: Foreign tourists riding bikes without helmets in Vietnam.  (https://www.facebook.com/journeywithjess2016/posts/when-you-are-trying-to-be-funny-but-then-realise-you-didnt-do-the-helmet-up-%EF%B8%8F-bi/1550193117112345/)


[generic] Extracting URL: https://www.facebook.com/journeywithjess2016/posts/when-you-are-trying-to-be-funny-but-then-reali...bi/1550193117112345/
[generic] 1550193117112345: Downloading webpage
[redirect] Following redirect to https://www.facebook.com/journeywithjess2016/videos/when-you-are-trying-to-be-funny-but-then-realise-you-didnt-do-the-helmet-up-%EF%B8%8F-bi/1683300813114029/
[facebook] Extracting URL: https://www.facebook.com/journeywithjess2016/videos/when-you-are-trying-to-be-funny-but-then-real...bi/1683300813114029/
[facebook] 1683300813114029: Downloading webpage
[info] 1683300813114029: Downloading 1 format(s): 1657898962311422v+1416909229541021a
[download] Destination: ./downloads/Foreign tourists riding bikes without helmets in Vietnam..f1657898962311422v.mp4
[download] 100% of   15.69MiB in 00:00:00 at 44.66MiB/s  
[download] Destination: ./downloads/Foreign tourists riding bikes without helmets in Vietnam..f1416909229541021a.m4a
[download] 100% of  175.52KiB in 00:00

17:28:06  INFO      Downloaded -> downloads/Foreign tourists riding bikes without helmets in Vietnam..mp4
17:28:06  INFO      ==================================================
17:28:06  INFO      PIPELINE COMPLETE
17:28:06  INFO        Search prompt    : helmet violation Vietnam motorcycle dashcam no helmet
17:28:06  INFO        Pages crawled    : 5
17:28:06  INFO        Videos discovered: 50
17:28:06  INFO        Downloads OK     : 47
17:28:06  INFO        Downloads failed : 3
17:28:06  INFO        Output directory : ./downloads
17:28:06  INFO      ==================================================


## 7. Results & Cleanup

In [9]:
import pandas as pd

if results:
    df = pd.DataFrame(results)
    print(df.to_string(index=False))
    print(f"\nSuccessful downloads: {(df['status'] == 'ok').sum()} / {len(df)}")
    print(f"Files saved in: {DOWNLOAD_DIR}")
    downloaded = list(Path(DOWNLOAD_DIR).glob("*"))
    for f in downloaded:
        print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")
else:
    print("No videos were discovered or downloaded.")

# Stop Ollama server
ollama_proc.terminate()
print("\nOllama server stopped.")

# Optionally remove downloaded videos to free disk space:
# import shutil
# shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
# print("Downloads cleaned up.")

17:28:07  INFO      NumExpr defaulting to 4 threads.


                                                                                                                 video_title                                                                                                                                                 raw_video_url                                                                                                                         description                                                                                                                             local_path status
                                           🇻🇳 My Helmet Fell Off… In The Middle of Traffic 😳 #vietnam #grabbike #베트남 #그랩오토바이                                                                                                                    https://www.youtube.com/shorts/RDHF2Yygmak                                                           A video showing a helmet falling off in the middle of Vietnamese traffic.                             